<a href="https://colab.research.google.com/github/KlauDCampos/Estudy-ML/blob/main/DataSet_Sem_t%C3%A9cnica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [ ]:
uploaded = files.upload()
arquivo = next(iter(uploaded))
df = pd.read_excel(arquivo)
print(df['label'].value_counts())
print(df.shape)

Saving 9.2.resultado_valoresModa_Multiclasse_excel_LabelEncoder_MinMaxScaler_label_binario.xlsx to 9.2.resultado_valoresModa_Multiclasse_excel_LabelEncoder_MinMaxScaler_label_binario (1).xlsx
label
0    444
1     60
Name: count, dtype: int64
(504, 21)


In [ ]:
df.head()


,sexo,estudante_profissional,cor_raca,faixa_etaria,portador_de_necessidade_especiais,estado_civil,filhos,meio_de_transporte,oriundo_escola,atualmente_reside,...,tipo_de_moradia,estado_civil_dos_pais,renda_mensal,participacao_economica,cota,formacao_nivel_medio,cursou_outro_nivel_superior,label,zona,codigo
0,0,0,0.2,0.50,0,0.5,0,1.000000,1.0,0.166667,...,1.0,0.0,0.25,0.50,0,0.0,0,0,0.932773,FGET9TJ3
1,1,0,0.8,0.50,0,0.0,1,0.666667,0.5,0.166667,...,0.5,0.0,0.50,0.25,0,0.0,0,0,0.638655,359HB47E
2,1,0,0.8,0.50,0,0.5,0,1.000000,0.0,0.666667,...,1.0,0.0,0.50,0.50,0,0.0,0,0,0.243697,W3IB5878
3,1,1,0.8,0.25,0,0.5,0,1.000000,0.5,0.833333,...,1.0,0.0,0.75,0.00,0,0.0,0,0,0.336134,L7D4XUHT
4,1,0,0.2,0.25,0,0.0,0,1.000000,1.0,0.166667,...,1.0,0.0,0.25,0.25,0,0.0,0,0,0.798319,H257IG5Y


In [ ]:

X,y = df.drop(['label','codigo'], axis=1), df['label']


smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42,test_size=0.3,stratify=y)
X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)
y_train_res_xgb = y_train_res - 1
y_test_xgb = y_test - 1

print(y_train_res.value_counts())
print(X_train_res.value_counts())

label
0    310
1    310
Name: count, dtype: int64
sexo  estudante_profissional  cor_raca  faixa_etaria  portador_de_necessidade_especiais  estado_civil  filhos  meio_de_transporte  oriundo_escola  atualmente_reside  acesso_ao_computador  tipo_de_moradia  estado_civil_dos_pais  renda_mensal  participacao_economica  cota  formacao_nivel_medio  cursou_outro_nivel_superior  zona    
1     0                       0.8       0.50          0                                  0.0           1       0.666667            1.0             0.833333           1                     1.0              0.00                   0.25          0.25                    0     0.0                   0                            0.890756    5
      1                       0.8       0.25          0                                  0.5           0       0.000000            0.5             0.666667           1                     1.0              0.00                   0.50          0.00                    0     0.0      

In [ ]:
pipe_tree = Pipeline([
    ('dtc', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ])
pipe_rf = Pipeline([
    ('rfc', RandomForestClassifier( n_estimators=150,random_state=42, max_depth=3))
])
pipe_knn = Pipeline([
    ('knn', KNeighborsClassifier(n_neighbors=4))
])
pipe_xgb = Pipeline([
    ('xgb', XGBClassifier(random_state=42,  eval_metric='mlogloss'))
])

In [ ]:
pipe_tree.fit(X_train_res,y_train_res)
pipe_rf.fit(X_train_res,y_train_res)
pipe_knn.fit(X_train_res,y_train_res)
pipe_xgb.fit(X_train_res,y_train_res)

print("Decision Tree")
print(pipe_tree.predict(X_test))
print("Random Forest")
print(pipe_rf.predict(X_test))
print("KNN")
print(pipe_knn.predict(X_test))
print("XGB")
print(pipe_xgb.predict(X_test))

Decision Tree
[0 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0
 0 1 0 0 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 1 1 1 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1
 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0
 0 1 1 0]
Random Forest
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0]
KNN
[0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0
 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 1 1 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 0
 1 1 1 0]
XGB
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [ ]:
print("Decision Tree")
print(classification_report(y_test,pipe_tree.predict(X_test)))
print("Random Forest")
print(classification_report(y_test,pipe_rf.predict(X_test)))
print("KNN")
print(classification_report(y_test,pipe_knn.predict(X_test)))
print("XGB")
print(classification_report(y_test,pipe_xgb.predict(X_test)))

Decision Tree
              precision    recall  f1-score   support

           0       0.90      0.78      0.84       134
           1       0.17      0.33      0.23        18

    accuracy                           0.73       152
   macro avg       0.53      0.56      0.53       152
weighted avg       0.81      0.73      0.76       152

Random Forest
              precision    recall  f1-score   support

           0       0.90      0.90      0.90       134
           1       0.22      0.22      0.22        18

    accuracy                           0.82       152
   macro avg       0.56      0.56      0.56       152
weighted avg       0.82      0.82      0.82       152

KNN
              precision    recall  f1-score   support

           0       0.89      0.81      0.84       134
           1       0.13      0.22      0.17        18

    accuracy                           0.74       152
   macro avg       0.51      0.51      0.51       152
weighted avg       0.80      0.74      0.7

In [ ]:
y_pred_treino_tree = pipe_tree.predict(X_train_res)
print("Decision Tree")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_tree))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_tree.predict(X_test)))
y_pred_treino_rf = pipe_rf.predict(X_train_res)
print("Random Forest")

print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_rf))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_rf.predict(X_test)))
y_pred_treino_knn = pipe_knn.predict(X_train_res)
print("KNN")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_knn))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_knn.predict(X_test)))
y_pred_treino_xgb = pipe_xgb.predict(X_train_res)
print("XGB")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_xgb))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_xgb.predict(X_test)))

Decision Tree
Acurácia no Treino: 0.8338709677419355
Acurácia no Teste:  0.7302631578947368
Random Forest
Acurácia no Treino: 0.8790322580645161
Acurácia no Teste:  0.8157894736842105
KNN
Acurácia no Treino: 0.9274193548387096
Acurácia no Teste:  0.7368421052631579
XGB
Acurácia no Treino: 1.0
Acurácia no Teste:  0.8486842105263158
